# Duplicates in reference items, need clean up

In [36]:
# Query to get the schema of the database
schema_query = "SELECT sql FROM sqlite_master WHERE type='table';"

# Execute the query
cursor = conn.cursor()
cursor.execute(schema_query)

# Fetch all results
schema_results = cursor.fetchall()

# Print the schema of each table
for schema in schema_results:
    print(schema[0])

CREATE TABLE "_prisma_migrations" (
    "id"                    TEXT PRIMARY KEY NOT NULL,
    "checksum"              TEXT NOT NULL,
    "finished_at"           DATETIME,
    "migration_name"        TEXT NOT NULL,
    "logs"                  TEXT,
    "rolled_back_at"        DATETIME,
    "started_at"            DATETIME NOT NULL DEFAULT current_timestamp,
    "applied_steps_count"   INTEGER UNSIGNED NOT NULL DEFAULT 0
)
CREATE TABLE "Receipt" (
    "id" INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
    "createdAt" DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    "updatedAt" DATETIME NOT NULL
)
CREATE TABLE sqlite_sequence(name,seq)
CREATE TABLE "Expense" (
    "id" INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
    "priceEach" REAL NOT NULL,
    "quantity" REAL NOT NULL,
    "receiptId" INTEGER NOT NULL,
    "receiptTextId" INTEGER,
    "productId" INTEGER,
    "createdAt" DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    "updatedAt" DATETIME NOT NULL,
    CONSTRAINT "Expense_receiptId_fke

In [ ]:
-- sqlite3

BEGIN TRANSACTION;

UPDATE Product
SET referenceItemId = (
    SELECT MIN(id) FROM ReferenceItem AS r WHERE r.name = ReferenceItem.name
)
WHERE referenceItemId IN (
    SELECT id FROM ReferenceItem WHERE id NOT IN (SELECT MIN(id) FROM ReferenceItem GROUP BY name)
);
DELETE FROM ReferenceItem
WHERE id NOT IN (SELECT MIN(id) FROM ReferenceItem GROUP BY name);

COMMIT;

^ i don't love the MIN practice, i prefer somehting like this (or use temporary mapping table):

In [ ]:
WITH ranked AS (
    SELECT id, name, updated_at,
           ROW_NUMBER() OVER (PARTITION BY name ORDER BY updated_at DESC) AS row_num
    FROM ReferenceItem
)
UPDATE Product
SET referenceItemId = (
    SELECT id FROM ranked WHERE row_num = 1
)
WHERE referenceItemId IN (SELECT id FROM ranked WHERE row_num > 1);

DELETE FROM referenceItemId WHERE id IN (SELECT id FROM ranked WHERE row_num > 1);

In [1]:
import pandas as pd
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect('dev-copy-1.db')

# Load the ReferenceItem table into a dataframe
df_reference_item = pd.read_sql_query("SELECT * FROM ReferenceItem", conn)

# Display the dataframe
df_reference_item

,id,name,quantity,unitOfMeasure,price,pricePerWeight,referenceUrl,createdAt,updatedAt
0,1,BBQ Sauce,455.0,g,2.47,0.005429,https://www.walmart.ca/en/ip/Kraft-BBQ-Sauce-L...,1726450695661,1735945682965
1,2,Beans,540.0,g,1.47,0.002722,https://www.walmart.ca/en/ip/Great-Value-Black...,1726450695726,1735945894924
2,3,Bouillon cubes,80.0,g,1.17,0.014625,https://www.walmart.ca/en/ip/Knorr-Beef-Flavou...,1726450695789,1735947821878
3,4,Candy,150.0,g,1.00,0.006667,-,1726450695851,1735947987572
4,5,Cereal,340.0,g,2.97,0.008735,https://www.walmart.ca/en/ip/Great-Value-Corn-...,1726450695914,1735947971754
5,6,Chia seeds,454.0,g,5.47,0.012048,https://walmart.ca/#missing,1726450695975,1726450695975
6,7,Chips,280.0,g,2.18,0.007786,https://walmart.ca/#missing,1726450696052,1726450696052
7,8,Chocolate,100.0,g,0.60,0.006000,https://walmart.ca/#missing,1726450696118,1726450696118
8,9,Chocolate chips,300.0,g,3.97,0.013233,https://walmart.ca/#missing,1726450696181,1726450696181
9,10,Cookies,70.0,g,0.67,0.009571,https://walmart.ca/#missing,1726450696245,1726450696245


In [31]:
df_reference_products_mapping = pd.read_csv('../data/reference_products_mapping.csv')
df_reference_products_mapping = df_reference_products_mapping[df_reference_products_mapping['id'].isin(df_reference_item['id'])]
df_reference_products_mapping.to_csv('updated_reference_products_mapping.csv', index=False)
# alternatively output with sql, but very small data size, this is quick, and I need csv file anyway

In [34]:
df_reference_item[['id','name']].to_csv('test.csv', index=False)

In [38]:
#  also remove, cooking oil (id: 43), Sugar (id: 44), Dates (id: 48), Peanut butter (id: 26), Milk (id: 20) # even alternative milks are mostly gf

In [ ]:
BEGIN TRANSACTION;

-- Delete expenses associated with the products
DELETE FROM Expense
WHERE productId IN (SELECT id FROM Product WHERE referenceItemId IN (43, 44, 48, 26, 20));

-- List of reference item IDs to delete
DELETE FROM Product
WHERE referenceItemId IN (43, 44, 48, 26, 20);

-- Delete the reference items themselves
DELETE FROM ReferenceItem
WHERE id IN (43, 44, 48, 26, 20);

COMMIT;